# Environment Setup

Before interacting with any AI model, we need to load our environment variables.

API keys and other sensitive credentials should never be hardcoded inside source code. Instead, they are stored in a `.env` file and loaded at runtime using the `python-dotenv` package.

This approach provides several advantages:

- Keeps credentials secure.
- Prevents accidental exposure when sharing notebooks.
- Simplifies environment management.
- Follows industry best practices for secret handling.

By calling `load_dotenv()`, all variables defined in the `.env` file become available to the current Python session.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Text input

# Creating a Multimodal AI Agent

In this example, we initialize an AI Agent using LangChain.

The agent is powered by the `gpt-5-nano` model and receives a custom system prompt that instructs it to behave as a science fiction writer.

```text
You are a science fiction writer, create a capital city at the users request.
```

System prompts play a critical role in shaping the behavior, tone, and expertise of a model. By modifying the instructions, we can transform the same underlying model into different types of assistants.

This agent will later be used to demonstrate how modern multimodal models can process not only text, but also images and audio.

In [2]:
from langchain.agents import create_agent

agent= create_agent(
    model= "gpt-5-nano",
    system_prompt= "You are a science fiction writer, create a capital city at the users request."
)

# Generating a Fictional Capital City

We begin by interacting with the model using text only.

The user's request is wrapped inside a `HumanMessage`, which represents a message coming from the user in LangChain's standardized messaging format.

The model receives the question:

```text
What is the capital of The Moon?
```

Based on the provided system prompt, the agent generates a creative science-fiction response rather than a factual answer.

This demonstrates how system prompts influence model behavior and how LangChain structures conversations using message objects.

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])
response= agent.invoke({"messages":[question]})

print(response["messages"][-1].content)

In this sci‑fi setting, the capital of The Moon is Lunopolis.

- Name: Lunopolis (also known by locals as Luna Prime in some texts)
- Location: Southern polar region, along the rim of Shackleton Crater. The city is tucked into sunlit corridors and shielded valleys carved by regolith, with skylights catching sunlight to power much of the city.
- Government: The Lunar Commonwealth’s seat of power. The Dome of Governance houses the Lunar Council, with the Archivists’ Conclave recording laws, histories, and scientific breakthroughs.
- Districts and features:
  - The Dome District: The political heart, where council chambers and courts reside.
  - The Glassways: A network of transit corridors and sky-lit boulevards, studded with schools, libraries, and museums.
  - The Icewell Quadrant: Water processing, ice mining control, and life-support infrastructure.
  - The Ether Gardens: Greenhouses and hydroponic farms that feed the city, kept lush by artificial sunlight.
  - The Helix Port: Spacep

## Image input

# Uploading an Image

To analyze visual content, we first need a way to provide images to the model.

The `FileUpload` widget allows users to upload files directly from the notebook interface.

In this example:

- Only image files are accepted.
- Multiple uploads are disabled.
- The uploaded image will later be converted into a format that can be consumed by the language model.

This forms the first step in enabling image understanding capabilities.

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.JPEG', multiple=False)
display(uploader)

FileUpload(value=(), accept='.JPEG', description='Upload')

# Inspecting Uploaded File Metadata

After uploading an image, we inspect the data returned by the upload widget.

The uploaded file contains several useful pieces of information:

- File name
- MIME type
- Size
- Binary content

Understanding this structure is important because multimodal models require the image data itself rather than just the file path.

The next step will extract the raw image bytes and prepare them for transmission to the model.

In [7]:
print(uploader.value)

({'name': 'WhatsApp Image 2026-05-11 at 18.59.14.jpeg', 'type': 'image/jpeg', 'size': 31993, 'content': <memory at 0x0000027E9C150340>, 'last_modified': datetime.datetime(2026, 6, 11, 20, 59, 52, 711000, tzinfo=datetime.timezone.utc)},)


# Converting an Image to Base64

Multimodal APIs typically expect image data to be transmitted in a portable format.

To achieve this, the uploaded image is converted into a Base64 encoded string.

The process consists of:

1. Extracting the binary image content.
2. Converting the memory buffer into bytes.
3. Encoding the bytes using Base64.
4. Producing a text representation that can be safely included inside API requests.

Base64 encoding allows binary files such as images to be embedded directly inside JSON payloads without requiring external file storage.

In [8]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

# Image Understanding with Multimodal Models

Modern multimodal models can process both text and visual information simultaneously.

In this example, the request contains:

- A text instruction.
- An image encoded in Base64 format.

The model receives both inputs as part of a single message.

This enables the model to reason about visual content, describe objects, analyze scenes, and generate responses that combine information from multiple modalities.

Unlike traditional language models that only process text, multimodal models can understand and reason about images directly.

In [9]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Here’s a capital that fits the image’s noir-meets-future vibe.

Capital name: Lysara Prime

Location and look:
- A city carved into the rim of a shattered moon, orbiting a pale-blue gas giant. Its rings cascade like staircases of light, and every level is a living layer cake of habitation: markets, governances, neighborhoods, and archives all stacked in concentric towers.
- The skyline is a field of fleur-de-lis motifs: spires shaped like stylized lilies, glass façades etched with heraldic lines, and ceremonial arches that glow at dusk with aurora-grade pigments.

Governance and society:
- The city is the capital of the Nebular Concord, a technocratic monarchy known as the Crowned Consensus. It blends tradition with real-time algorithmic governance, where votes are converted into resonance signals that guide policy in the city’s neural lattice.
- A ruling body called the Scepter Council sits beneath the “Crown,” a colossal transparent dome that houses the city’s central data cathedral.

## Audio input

# Recording Audio Input

Multimodal systems are not limited to text and images. Many modern models can also process audio.

In this section, we record a short audio clip using the local microphone.

The recording pipeline includes:

- Capturing audio samples.
- Monitoring recording progress.
- Waiting for recording completion.
- Storing the audio in WAV format.

The resulting audio will later be converted into a format suitable for transmission to the model.

This demonstrates how AI systems can interact with spoken language in addition to written text.

In [10]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.79it/s]

Done.


# Audio Understanding with Multimodal Models

The recorded audio is encoded and provided to a multimodal model alongside a text instruction.

The request contains:

- A textual question.
- An audio file represented as Base64.
- Metadata describing the audio format.

The model is then asked to analyze the audio content and generate a response.

This workflow demonstrates one of the most powerful capabilities of modern AI systems: the ability to understand and reason across multiple modalities including text, images, and speech.

> Note:
> Availability of audio-capable models may vary depending on API access levels and provider support. Some preview models may become deprecated or restricted over time.

In [11]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-4o-audio-preview` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

# Key Takeaways

This notebook demonstrates the evolution from traditional text-based interactions to fully multimodal AI systems.

Throughout the notebook we explored:

- Text generation using AI agents.
- System prompts and role customization.
- Image upload and preprocessing.
- Base64 encoding for media transmission.
- Image understanding and reasoning.
- Audio recording and processing.
- Audio understanding using multimodal models.

Modern AI applications increasingly combine multiple input modalities within a single workflow, enabling richer user experiences and more powerful reasoning capabilities.

Understanding how to prepare, encode, and transmit different data types is a fundamental skill when building production-ready multimodal AI applications.